# Distil mBERT Fine-Tuning

This notebook benchmarks `distilbert-base-multilingual-cased` with LoRA for binary scam detection. The objective is to test whether a multilingual distilled encoder can retain high recall while remaining deployable.

In [2]:
# Training, PEFT, and experiment tracking dependencies

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datasets

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient


## 1. Load the Prepared Training Set

I reuse the balanced dataset generated in the preparation notebook so each model comparison starts from the same train/test foundation.

In [40]:
# Load Data
training_set_path = "../data/processed/train_ds/data-00000-of-00001.arrow"
ds = datasets.Dataset.from_file(training_set_path)
ds.to_pandas().head(10)

,text,labels
0,the original online source for market research...,fraud
1,last up to 4 hours with 100 % natural viagra !...,fraud
2,doctor discovers s ' perm increasement pill\n\...,fraud
3,good time after party : )\n\nthe latest invent...,fraud
4,Important notice: Your account verification re...,fraud
5,Claim a free beauty product! today and enjoy e...,ham
6,FROM 88066 LOST £12 HELP\n,fraud
7,business relationship\n\ni am engineer mr duke...,fraud
8,rolex watches now for pea nut\n\nhows it been ...,fraud
9,"caller: Good afternoon, your catering order fo...",ham


## 2. Create a Validation Split

The holdout validation set is used to monitor training quality and select the best checkpoint based on recall.

In [44]:
# Split the training data again so model selection uses a clean validation subset
ds_split = ds.train_test_split(test_size=0.1, seed=42)

ds_train = ds_split['train']
ds_val = ds_split['test']

print(f"Train Set: {len(ds_train)}, {ds_train.to_pandas()['labels'].value_counts()}")
print(f"Validation Set: {len(ds_val)}, {ds_val.to_pandas()['labels'].value_counts()}")


Train Set: 13307, labels
ham      6704
fraud    6603
Name: count, dtype: int64
Validation Set: 1479, labels
fraud    771
ham      708
Name: count, dtype: int64


## 3. Estimate Sequence Length

I inspect the 90th percentile token length to choose a truncation limit that is large enough for most messages but still cost-aware.

In [45]:
lengths = [len(x.split()) for x in ds_train["text"]]
print(f"90th percentile length: {np.percentile(lengths, 90)}")

90th percentile length: 222.0


## 4. Encode Labels and Tokenize Text

The labels are converted to integers and the text is tokenized with the Distil mBERT tokenizer using a 256-token cap.

In [50]:
# Convert string labels into integer targets expected by the trainer
str2int = {'fraud': 1, 'ham': 0}

encoded_ds_train = ds_train.map(lambda x: {'labels': str2int[x['labels']]})
encoded_ds_val = ds_val.map(lambda x: {'labels': str2int[x['labels']]})

# Tokenize with a fixed maximum length to keep training predictable on local hardware
distilbert_model_id = 'distilbert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(distilbert_model_id)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

tokenized_ds_train = encoded_ds_train.map(tokenize_function, batched=True)
tokenized_ds_val = encoded_ds_val.map(tokenize_function, batched=True)

print(tokenized_ds_train.features)
print(f'Train Set:{tokenized_ds_train}\n')
print(tokenized_ds_val.features)
print(f'Validation Set:{tokenized_ds_val}\n')


{'text': Value('string'), 'labels': Value('int32'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'token_type_ids': List(Value('int8'))}
Train Set:Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask', 'token_type_ids'],
    num_rows: 13307
})

{'text': Value('string'), 'labels': Value('int32'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'token_type_ids': List(Value('int8'))}
Validation Set:Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask', 'token_type_ids'],
    num_rows: 1479
})



## 5. Apply LoRA and Fine-Tune

Only a small set of low-rank adapters is trained instead of updating the full base model. This keeps experimentation feasible on a Mac while preserving strong classification performance.

In [55]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

distilbert_model_id = 'distilbert-base-multilingual-cased'
bert_base = AutoModelForSequenceClassification.from_pretrained(distilbert_model_id, num_labels=2)

# Add LoRA adapters instead of fine-tuning every weight
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias='none',
    task_type='SEQ_CLS',
    target_modules=['q_lin', 'k_lin', 'v_lin', 'out_lin'],
)
lora_distilmbert = get_peft_model(bert_base, lora_config)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted_class = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predicted_class),
        'precision': precision_score(labels, predicted_class),
        'recall': recall_score(labels, predicted_class),
        'f1_score': f1_score(labels, predicted_class),
    }

training_args = TrainingArguments(
    output_dir='./results_lora_distil-mBert',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_strategy='best',
    greater_is_better=True,
    load_best_model_at_end=True,
    metric_for_best_model='recall',
    fp16=False,
    bf16=True,
)

trainer_lora_distilmBERT = Trainer(
    model=lora_distilmbert,
    args=training_args,
    train_dataset=tokenized_ds_train,
    eval_dataset=tokenized_ds_val,
    compute_metrics=compute_metrics,
)

trainer_lora_distilmBERT.train()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 136,213,252 || trainable%: 0.6512


/opt/miniconda3/envs/dlenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.294989
1000,0.155425
1500,0.108290
2000,0.089680
2500,0.088402
3000,0.075062


TrainOutput(global_step=3328, training_loss=0.12965406477451324, metrics={'train_runtime': 9168.6455, 'train_samples_per_second': 5.805, 'train_steps_per_second': 0.363, 'total_flos': 3598010312171520.0, 'train_loss': 0.12965406477451324, 'epoch': 4.0})

## 6. Evaluate on the Validation Split

These metrics tell me whether the adapter-tuned model is strong enough to keep as a deployment candidate.

In [56]:
lora_distilmBERT_results = trainer_lora_distilmBERT.evaluate()

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1 Score
0.075062,0.099909,3328,0.968222,0.977573,0.961089,0.969261


In [58]:
print("LORA Finetuning Results:")
print(lora_distilmBERT_results)

LORA Finetuning Results:
{'eval_loss': 0.09990882128477097, 'eval_accuracy': 0.9682217714672076, 'eval_precision': 0.9775725593667546, 'eval_recall': 0.9610894941634242, 'eval_f1_score': 0.9692609548724657}


## 7. Log the Model to MLflow

The final training step stores the tokenizer, adapter weights, and run metadata in MLflow so I can compare experiments cleanly later.

In [59]:
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://localhost:8080")
mlflow.set_experiment("scam-detector-finetuning")

with mlflow.start_run(run_name="lora_distilmBERT_v2") as run:
        # 1. Log params
        mlflow.log_params(training_lora_distilmBERT_args.to_dict())
            
        # 2. Log metrics
        mlflow.log_metrics(lora_distilmBERT_results)

        # 3. Log model + tokenizer
        mlflow.transformers.log_model(
        transformers_model={
                    "model":     trainer_lora_distilmBERT.model,       # model 
                    "tokenizer": tokenizer # tokenizer 
                },
                artifact_path="lora_distilmBERT_v2",
                task="text-classification",
            )

# Verifikasi model yang sudah di log
client = MlflowClient(tracking_uri="http://localhost:8080")
experiment = client.get_experiment_by_name("scam-detector-finetuning")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
print(f"Runs in experiment 'scam-detector-finetuning': {[run.info.run_id for run in runs]}")


python(59284) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59285) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
2026/05/09 18:22:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 18:22:47 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026/05/09 18:22:51 INFO mlflow.transformers: Skipping saving pretrained model weights to disk as the save_pretrained argumentis set to False. The reference to the HuggingFace Hub repository distilbert-base-multilingual-cased will be logged instead.
2026/05/09 18:22:52 INFO mlflow.transformers: A local checkpoint path or PEFT model is given as the `transformers_model`. To avoid loading the full model into m

🏃 View run lora_distilmBERT_v2 at: http://localhost:8080/#/experiments/1/runs/bf1a1dd415d741baa0783d07e4dbaca6
🧪 View experiment at: http://localhost:8080/#/experiments/1
Runs in experiment 'scam-detector-finetuning': ['bf1a1dd415d741baa0783d07e4dbaca6', '4b367643a71244478d7946e4f7685d2d', '0783e27cccd94747bec51d71e5731cd8', '149c470308ad4d4ea886051b09346fd6', 'd7f0b29667e94ccb99f8bb5e7050d9a8', 'f3315e08d68f45a583bd77f0959a4f50', 'db87fe9108a944e3a171d2468e2764c6', '7a2682476fab4ad49d8dac6a1901874a']
